# 문서 근거 업무 챗봇 — Gradio + OpenAI

**현장 데이터 디지털화 과정 · Day05 코랩 실습 예시 (RAG 기반 사내 지식 자산화)**

업무 파일을 최대 5개 올리면, 그 문서를 읽고 **근거(파일 · 위치 · 원문)** 를 붙여 답하는 챗봇을 만듭니다.
답 뒤에는 이어서 물어볼 질문 1~3개가 **단추**로 나오고, 단추를 누르면 그 질문에 바로 답합니다.

### 시작하는 방법
1. OpenAI API 키를 준비합니다(수업 시간에 나눠 드립니다).
   - 왼쪽 열쇠 모양 **「보안 비밀」** 에 이름 `OPENAI_API_KEY` 로 넣고 「노트북 액세스」를 켜 두면 매번 붙여 넣지 않아도 됩니다.
   - 안 넣었다면 챗봇 화면의 「API 키」 칸에 붙여 넣으면 됩니다.
2. 위에서부터 셀을 하나씩 실행합니다(셀 왼쪽 ▶ 또는 `Shift + Enter`).
3. 마지막 셀을 실행하면 아래에 챗봇 화면이 뜹니다. `https://….gradio.live` 링크를 누르면 새 창에서 크게 열립니다.
4. 처음에는 **② 예제 문서** 로 연습하고, 그다음 내 업무 파일로 바꿔 봅니다.

## 이 챗봇을 AI 에게 만들어 달라고 할 때 쓴 프롬프트

처음 프롬프트로도 화면은 나오지만, 몇 가지가 빠져 있으면 코랩에서 오류가 납니다.
아래는 빠진 것을 채운 프롬프트입니다. `[대괄호]` 를 내 업무에 맞게 바꿔 붙여 넣어 보세요.

```text
너는 파이썬 Gradio 개발자야. 구글 코랩에서 바로 실행되는 「문서 근거 업무 챗봇」을 만들어줘.

[하는 일]
- 내가 업무 파일을 최대 5개 올리면 내용을 읽고, 질문에 답할 때 반드시 근거(파일 이름 · 쪽이나 시트 위치 · 원문 한 구절)를 붙여줘.
- 문서에 없는 내용은 「문서에서 찾지 못했습니다」라고 답하고 추측하지 마.
- 답 뒤에는 이어서 물어볼 질문 1~3개를 단추로 보여 주고, 단추를 누르면 그 질문에 바로 답해줘.
- 화면 맨 위에 「시작하는 방법」을 1~4단계로 보여줘.

[파일]
- pdf · docx · xlsx · pptx · hwpx · txt · csv 를 읽어줘. 글자만 읽지 말고 형식에 맞는 라이브러리(pypdf · python-docx · openpyxl · python-pptx)로 읽어줘.
- .hwp 는 읽지 말고 「HWPX 나 PDF 로 저장해 올려 주세요」라고 안내해줘.
- 문서가 길면 조각으로 나누고, 질문과 가까운 조각만 골라 보내줘(OpenAI 임베딩).

[기술 조건]
- 최신 Gradio(6 버전)에서 돌아가게 해줘. 대화 기록은 {"role": ..., "content": ...} 형식으로 쓰고, 튜플 형식은 쓰지 마.
- OpenAI 는 openai 1.x 의 OpenAI() 클라이언트와 [gpt-4.1-mini] 모델을 써줘. API 키는 화면 입력칸 또는 코랩 보안 비밀 OPENAI_API_KEY 에서 읽어줘.
- 답은 JSON(answer · evidence · followups)으로 받아서 근거와 꼬리 질문을 안정적으로 나눠줘.
- 오류가 나면 화면에 한국어로 무엇을 하면 되는지 알려줘. 함수가 돌려주는 값의 개수는 어떤 경우에도 outputs 개수와 같게 해줘.

[대상] 개발을 처음 해 보는 [건설사 실무자]가 씁니다. 화면 글자는 모두 한국어로 짧게 써줘.
```

### 처음 코드가 오류 난 이유 (같은 오류를 만나면 여기부터 보세요)
| 증상 | 원인 | 고친 방법 |
|---|---|---|
| 질문하면 `Data incompatible with messages format` 같은 오류 | 지금 코랩에 깔리는 Gradio 6 은 대화 기록을 `{"role", "content"}` 형식만 받습니다. 옛 코드는 `(질문, 답)` 튜플로 넣었습니다 | 대화 기록을 role · content 형식으로 저장 |
| 키를 안 넣거나 질문이 비면 `returned too few output values` | 오류일 때 돌려주는 값 개수가 화면 출력 개수와 달랐습니다 | `gr.Error(...)` 로 안내 창을 띄움 |
| PDF · 워드 · 엑셀을 올리면 답이 엉뚱함 | 모든 파일을 글자(txt)로 열어 깨진 글자를 읽었습니다 | 형식마다 맞는 라이브러리로 읽음 |
| 긴 문서에서 뒤쪽 내용을 모름 | 파일마다 앞 4,000자만 보냈습니다 | 조각으로 나누고 질문과 가까운 조각을 골라 보냄 |

## ① 필요한 라이브러리 설치 (1분쯤 걸립니다)

In [ ]:
!pip install -q -U gradio openai pypdf python-docx openpyxl python-pptx
import gradio, openai
print("Gradio", gradio.__version__, "· openai", openai.__version__)

## ② 예제 문서 만들기
연습용 건설 현장 문서 4개(계약서 · 공정표 · 회의록 · 안전점검)를 `/content/예제문서` 에 만듭니다. 모두 지어낸 내용입니다.
챗봇 화면의 「또는 구글 드라이브 폴더에서 읽기」 → 「폴더에서 읽기」를 누르면 바로 읽힙니다.

In [ ]:
import os, docx, openpyxl
os.makedirs("/content/예제문서", exist_ok=True)

d = docx.Document()
d.add_heading("공사 도급 계약서 (예제)", 1)
for t in ["공사명: 한빛 물류센터 증축공사",
          "공사 기간: 2026년 7월 1일부터 2026년 12월 20일까지",
          "계약 금액: 48억 원(부가가치세 포함)",
          "지체상금: 준공이 늦어지면 하루마다 계약 금액의 1,000분의 0.5 를 낸다.",
          "설계 변경은 발주자와 서면으로 합의한 뒤에만 할 수 있다."]:
    d.add_paragraph(t)
d.save("/content/예제문서/계약서.docx")

wb = openpyxl.Workbook(); ws = wb.active; ws.title = "공정표"
ws.append(["공종", "시작", "종료", "진척률(%)", "비고"])
for r in [["토공", "2026-07-01", "2026-07-31", 100, ""],
          ["골조", "2026-08-01", "2026-10-15", 85, "우천으로 1주 지연"],
          ["마감", "2026-10-16", "2026-12-05", 10, ""],
          ["준공검사", "2026-12-08", "2026-12-20", 0, ""]]:
    ws.append(r)
wb.save("/content/예제문서/공정표.xlsx")

open("/content/예제문서/주간회의록_1105.txt", "w", encoding="utf-8").write(
"주간 공정회의 (2026-11-05)\n"
"1. 골조 공정이 우천으로 1주 늦어졌다. 마감 공정 인원을 6명에서 9명으로 늘려 만회한다.\n"
"2. 3층 슬래브 균열 2곳 발견. 구조기술사 검토를 11월 12일까지 받는다.\n"
"3. 다음 회의: 11월 12일 오전 9시 현장 사무실.\n")

open("/content/예제문서/안전점검.txt", "w", encoding="utf-8").write(
"안전점검 결과 (2026-11-03)\n"
"- 2층 개구부 안전난간 일부 없음 → 11월 4일까지 설치 (담당: 공무팀)\n"
"- 타워크레인 와이어로프 마모 → 교체 완료\n"
"- 근로자 안전모 미착용 3건 적발 → 아침 조회 때 다시 교육\n")

os.environ["DOC_FOLDER"] = "/content/예제문서"
print(sorted(os.listdir("/content/예제문서")))

## ③ (선택) 구글 드라이브 연결 — 내 업무 파일을 드라이브에서 읽을 때
실행하면 구글 계정 허용 창이 뜹니다. 수업에서 쓰는 계정으로 허용하세요.
`DOC_FOLDER` 를 내 폴더 경로로 바꾸면 챗봇의 「폴더에서 읽기」 칸에 그 경로가 미리 들어갑니다.
파일을 화면에서 직접 올릴 거라면 이 셀은 건너뛰어도 됩니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["DOC_FOLDER"] = "/content/drive/MyDrive/[업무문서 폴더]"   # ← 내 폴더 이름으로 바꾸기
print(os.environ["DOC_FOLDER"], "있음" if os.path.isdir(os.environ["DOC_FOLDER"]) else "← 이 폴더가 없습니다. 이름을 확인하세요")

## ④ API 키 불러오기
코랩 「보안 비밀」에 `OPENAI_API_KEY` 를 넣었다면 여기서 읽어 둡니다. 없으면 챗봇 화면에 직접 넣으면 됩니다.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("보안 비밀에서 API 키를 읽었습니다.")
except Exception:
    print("보안 비밀에 키가 없습니다. 챗봇 화면의 「API 키」 칸에 붙여 넣으세요.")

## ⑤ 챗봇 실행
실행하면 아래에 화면이 뜹니다. 멈추려면 셀의 ■(중지)를 누르세요. 코드를 고쳤다면 이 셀만 다시 실행하면 됩니다.

해 볼 질문: 「준공일과 지체상금 조건을 알려줘」 · 「공정이 늦어진 이유와 만회 계획은?」 · 「아직 조치가 안 끝난 안전 지적 사항은?」

In [ ]:
# ============================================================
# 문서 근거 업무 챗봇 (Gradio + OpenAI)
#  - 파일 최대 5개를 올리면 내용을 읽어 「근거」와 함께 답합니다.
#  - 답 뒤에 이어서 물어볼 질문 1~3개를 단추로 보여 줍니다.
#    단추를 누르면 그 질문에 바로 답합니다.
# ============================================================
import os, re, json, html, zipfile, inspect
import gradio as gr
from openai import OpenAI

MAX_FILES = 5
CHUNK_CHARS = 1200           # 근거 조각 한 개의 길이
ALL_IN_LIMIT = 60000         # 문서 전체가 이보다 짧으면 통째로 보냅니다
TOP_K = 12                   # 길면 질문과 가까운 조각 12개만 보냅니다


# ---------- 1. 파일 읽기 (형식마다 위치 정보를 붙여 둡니다) ----------
def read_pdf(path):
    from pypdf import PdfReader
    return [(f"{i}쪽", p.extract_text() or "") for i, p in enumerate(PdfReader(path).pages, 1)]

def read_docx(path):
    import docx
    d = docx.Document(path)
    parts = [p.text for p in d.paragraphs if p.text.strip()]
    for t in d.tables:
        for r in t.rows:
            parts.append(" | ".join(c.text.strip() for c in r.cells))
    # 문단 20개씩 묶어 「문단 1~20」처럼 위치를 붙입니다
    return [(f"문단 {i+1}~{min(i+20, len(parts))}", "\n".join(parts[i:i+20])) for i in range(0, len(parts), 20)]

def read_xlsx(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    out = []
    for ws in wb.worksheets:
        rows = [" | ".join("" if v is None else str(v) for v in r)
                for r in ws.iter_rows(values_only=True) if any(v is not None for v in r)]
        for i in range(0, len(rows), 40):
            out.append((f"시트 {ws.title} {i+1}~{min(i+40, len(rows))}행", "\n".join(rows[i:i+40])))
    return out

def read_pptx(path):
    from pptx import Presentation
    out = []
    for i, s in enumerate(Presentation(path).slides, 1):
        t = [sh.text_frame.text for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()]
        out.append((f"{i}번 슬라이드", "\n".join(t)))
    return out

def read_hwpx(path):
    out = []
    with zipfile.ZipFile(path) as z:
        secs = sorted(n for n in z.namelist() if re.match(r"Contents/section\d+\.xml", n))
        for n in secs:
            xml = z.read(n).decode("utf-8", "ignore")
            # 글자는 <hp:t> 안에만 있습니다 (필드 · 링크 설정값은 빼고 읽습니다)
            paras = [html.unescape("".join(re.findall(r"<hp:t(?:\s[^>]*)?>(.*?)</hp:t>", p, flags=re.S)))
                     for p in re.findall(r"<hp:p[ >].*?</hp:p>", xml, flags=re.S)]
            paras = [re.sub(r"<[^>]+>", "", p) for p in paras]
            out.append((n.split("/")[-1].replace(".xml", ""), "\n".join(p for p in paras if p.strip())))
    return out

def read_text(path):
    for enc in ("utf-8", "cp949"):
        try:
            with open(path, encoding=enc) as f:
                return [("본문", f.read())]
        except UnicodeDecodeError:
            continue
    raise ValueError("글자 인코딩을 알 수 없습니다")

READERS = {".pdf": read_pdf, ".docx": read_docx, ".xlsx": read_xlsx, ".pptx": read_pptx,
           ".hwpx": read_hwpx, ".txt": read_text, ".md": read_text, ".csv": read_text}


def make_chunks(filename, parts):
    chunks = []
    for loc, text in parts:
        text = text.strip()
        for i in range(0, len(text), CHUNK_CHARS):
            chunks.append({"file": filename, "loc": loc, "text": text[i:i+CHUNK_CHARS]})
    return chunks


# ---------- 2. 상태: 읽은 조각 · 임베딩 ----------
DOCS = {"chunks": [], "vectors": None}

def get_client(key_box):
    key = (key_box or "").strip() or os.environ.get("OPENAI_API_KEY", "")
    if not key:
        raise gr.Error("OpenAI API 키를 넣어 주세요 (위 「API 키」 칸).")
    return OpenAI(api_key=key)

def load_files(files):
    DOCS["chunks"], DOCS["vectors"] = [], None
    if not files:
        return "파일을 1~5개 올린 뒤 「문서 읽기」를 눌러 주세요."
    lines = []
    if len(files) > MAX_FILES:
        lines.append(f"- 파일이 {len(files)}개라 앞의 {MAX_FILES}개만 읽습니다.")
    for f in files[:MAX_FILES]:
        path = f if isinstance(f, str) else f.name
        name = os.path.basename(path)
        ext = os.path.splitext(name)[1].lower()
        if ext == ".hwp":
            lines.append(f"- ❌ {name}: .hwp 는 읽을 수 없습니다. 한글에서 HWPX 또는 PDF 로 저장해 올려 주세요.")
            continue
        reader = READERS.get(ext)
        if not reader:
            lines.append(f"- ❌ {name}: 지원하지 않는 형식입니다 (pdf · docx · xlsx · pptx · hwpx · txt · md · csv).")
            continue
        try:
            ch = make_chunks(name, reader(path))
        except Exception as e:
            lines.append(f"- ❌ {name}: 읽기 오류 ({e})")
            continue
        n = sum(len(c["text"]) for c in ch)
        if n == 0:
            lines.append(f"- ⚠️ {name}: 글자가 없습니다 (스캔 PDF 일 수 있습니다).")
            continue
        DOCS["chunks"] += ch
        lines.append(f"- ✅ {name}: {n:,}자 · 조각 {len(ch)}개")
    total = sum(len(c["text"]) for c in DOCS["chunks"])
    lines.append(f"\n**모두 {total:,}자.** 이제 오른쪽에 질문을 적어 보세요.")
    return "\n".join(lines)


def load_folder(folder):
    # 구글 드라이브(또는 코랩) 폴더에서 읽을 수 있는 파일을 앞에서부터 5개 고릅니다
    folder = (folder or "").strip()
    if not folder or not os.path.isdir(folder):
        return f"폴더를 찾을 수 없습니다: {folder}\n(드라이브 연결 셀을 먼저 실행했는지 확인해 주세요)"
    names = sorted(n for n in os.listdir(folder) if os.path.splitext(n)[1].lower() in READERS)
    if not names:
        return "이 폴더에 읽을 수 있는 파일이 없습니다."
    return load_files([os.path.join(folder, n) for n in names])


def pick_chunks(client, question):
    ch = DOCS["chunks"]
    if sum(len(c["text"]) for c in ch) <= ALL_IN_LIMIT:
        return ch
    if DOCS["vectors"] is None:          # 처음 한 번만 조각마다 임베딩을 만듭니다
        vecs = []
        for i in range(0, len(ch), 100):
            r = client.embeddings.create(model="text-embedding-3-small",
                                         input=[c["text"] for c in ch[i:i+100]])
            vecs += [d.embedding for d in r.data]
        DOCS["vectors"] = vecs
    q = client.embeddings.create(model="text-embedding-3-small", input=[question]).data[0].embedding
    score = [sum(a*b for a, b in zip(q, v)) for v in DOCS["vectors"]]
    best = sorted(range(len(ch)), key=lambda i: -score[i])[:TOP_K]
    return [ch[i] for i in sorted(best)]


# ---------- 3. 프롬프트 ----------
SYSTEM_PROMPT = """너는 사용자가 올린 문서만 근거로 답하는 업무용 챗봇이야.
규칙:
1. 답은 아래 [문서] 안의 내용으로만 해. 문서에 없는 내용은 「문서에서 찾지 못했습니다」라고 말하고 추측하지 마.
2. 답의 핵심 문장마다 근거를 붙여. 근거는 파일 이름 · 위치(쪽 · 시트 · 문단) · 원문 한 구절로 적어.
3. 여러 문서의 내용이 서로 다르면 다르다고 밝히고 각각의 근거를 보여 줘.
4. 답은 한국어로, 업무 보고처럼 짧고 분명하게 써. 필요하면 번호 목록을 써.
5. 답 뒤에, 방금 답을 이어서 더 깊이 들어갈 질문을 1~3개 만들어 줘. 문서로 답할 수 있는 질문만 만들어.
반드시 아래 JSON 하나로만 답해:
{"answer": "답 본문(마크다운)",
 "evidence": [{"file": "파일 이름", "loc": "위치", "quote": "원문 구절"}],
 "followups": ["이어서 물어볼 질문 1", "질문 2", "질문 3"]}"""


def to_text(content):
    # Gradio 6 은 대화 내용을 [{"type": "text", "text": ...}] 목록으로 돌려줍니다
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content)


def ask(key_box, model, question, history):
    question = (question or "").strip()
    if not question:
        raise gr.Error("질문을 적어 주세요.")
    if not DOCS["chunks"]:
        raise gr.Error("먼저 왼쪽에 파일을 올리고 「문서 읽기」를 눌러 주세요.")
    client = get_client(key_box)
    ctx = "\n\n".join(f"<<{c['file']} · {c['loc']}>>\n{c['text']}" for c in pick_chunks(client, question))
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for m in history[-6:]:                       # 앞 대화 3번까지 기억합니다
        msgs.append({"role": m["role"], "content": to_text(m["content"])})
    msgs.append({"role": "user", "content": f"[문서]\n{ctx}\n\n[질문]\n{question}"})
    r = client.chat.completions.create(model=model, messages=msgs,
                                       response_format={"type": "json_object"})
    raw = r.choices[0].message.content
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = {"answer": raw, "evidence": [], "followups": []}
    text = data.get("answer", "").strip()
    ev = data.get("evidence") or []
    if ev:
        text += "\n\n**근거**\n" + "\n".join(
            f"- 📄 {e.get('file','')} · {e.get('loc','')} — 「{e.get('quote','')}」" for e in ev)
    fups = [q for q in (data.get("followups") or []) if isinstance(q, str) and q.strip()][:3]
    history = history + [{"role": "user", "content": question}, {"role": "assistant", "content": text}]
    btns = [gr.update(value=fups[i], visible=True) if i < len(fups) else gr.update(visible=False)
            for i in range(3)]
    return ("", history, *btns)


# ---------- 4. 화면 ----------
HOWTO = """### 시작하는 방법
1. **API 키** 칸에 OpenAI API 키(`sk-...`)를 붙여 넣습니다. (코랩 「보안 비밀」에 `OPENAI_API_KEY` 로 넣어 두었다면 비워 두어도 됩니다)
2. 왼쪽에 업무 파일을 **최대 5개** 올리고 **「문서 읽기」** 를 누릅니다. (pdf · docx · xlsx · pptx · hwpx · txt · md · csv)
3. 오른쪽 아래에 질문을 적고 Enter 를 누릅니다. 답 아래에 **근거(파일 · 위치 · 원문)** 가 붙습니다.
4. 답 밑의 **이어서 물어보기** 단추를 누르면 그 질문에 바로 답합니다.

예시 질문: 「이 문서들의 핵심 내용을 5줄로 정리해줘」 · 「공사 기간과 마감일이 적힌 곳을 모두 찾아줘」"""

chat_kwargs = {"label": "업무 대화", "height": 460}
if "type" in inspect.signature(gr.Chatbot.__init__).parameters:   # Gradio 4~5 는 messages 형식을 직접 지정
    chat_kwargs["type"] = "messages"

with gr.Blocks(title="문서 근거 업무 챗봇") as demo:
    gr.Markdown("# 문서 근거 업무 챗봇")
    gr.Markdown(HOWTO)
    with gr.Row():
        key_box = gr.Textbox(label="API 키", type="password", placeholder="sk-...", scale=3)
        model = gr.Dropdown(["gpt-4.1-mini", "gpt-4o-mini", "gpt-4.1"], value="gpt-4.1-mini",
                            label="모델", scale=1)
    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(label=f"업무 파일 (최대 {MAX_FILES}개)", file_count="multiple", type="filepath")
            load_btn = gr.Button("문서 읽기", variant="primary")
            with gr.Accordion("또는 구글 드라이브 폴더에서 읽기", open=False):
                folder_box = gr.Textbox(label="폴더 경로", value=os.environ.get("DOC_FOLDER", ""),
                                        placeholder="/content/drive/MyDrive/업무문서")
                folder_btn = gr.Button("폴더에서 읽기")
            status = gr.Markdown("아직 읽은 문서가 없습니다.")
        with gr.Column(scale=2):
            chat = gr.Chatbot(**chat_kwargs)
            q_box = gr.Textbox(label="질문", placeholder="문서에 대해 궁금한 것을 적고 Enter")
            gr.Markdown("**이어서 물어보기**")
            with gr.Row():
                fbtns = [gr.Button(visible=False, size="sm") for _ in range(3)]
            clear = gr.Button("대화 지우기", size="sm")

    outs = [q_box, chat, *fbtns]
    load_btn.click(load_files, files, status)
    folder_btn.click(load_folder, folder_box, status)
    q_box.submit(ask, [key_box, model, q_box, chat], outs)
    for b in fbtns:                      # 단추의 글자(=질문)를 그대로 질문으로 보냅니다
        b.click(ask, [key_box, model, b, chat], outs)
    clear.click(lambda: ([], *[gr.update(visible=False)] * 3), None, [chat, *fbtns])

if __name__ == "__main__":
    demo.launch(share=True, debug=True)